# 02 — Run evaluation

Evaluates every enabled agent in `aql_agents` against its question bank using the official
`fabric-data-agent-sdk`. Appends verdicts to `evaluation_output` (+`_steps`), records run metadata
in `aql_eval_runs`, and rebuilds `aql_agent_scores`.

One failing agent never aborts the run — the failure is recorded with its error.

> Requires the *agents'* workspaces to be on a **paid F SKU** capacity (trial FTL SKUs return 403).

In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "{{DEFAULT_LAKEHOUSE_NAME}}",
        "id": "{{DEFAULT_LAKEHOUSE_ID}}",
        "workspaceId": "{{WORKSPACE_ID}}"
    }
}

In [ ]:
# PARAMETERS -----------------------------------------------------------------
AGENT_FILTER = ""        # only agents whose name contains this (empty = all)
BANK_FILTER = ""         # only this bank (empty = all)
SAMPLE_SIZE = 0          # cap questions per agent (0 = all)
CRITIC_PROMPT = ""       # custom LLM-judge prompt with {query}/{expected_answer}/{actual_answer}
NUM_QUERY_REPEATS = 1    # ask each question N times (consistency testing)
MAX_WORKERS = 5          # parallel questions per agent
OUTPUT_TABLE = "evaluation_output"
REGRESSION_THRESHOLD = 0.10  # alert when accuracy drops >= this vs previous run
FAIL_ON_REGRESSION = False   # True: fail this notebook (and pipeline run) on regression


In [ ]:
# Install pinned SDK + defensive import (works in notebook, pipeline and Livy) -
import subprocess, sys
SDK_PIN = "fabric-data-agent-sdk==0.1.26a0"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", SDK_PIN, "-U", "typing_extensions"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("pip install failed: " + r.stderr[-1500:])

def _import_sdk():
    try:
        from fabric.dataagent.evaluation import evaluate_data_agent
        return evaluate_data_agent
    except ImportError:
        # Fabric runtime pre-imports an old typing_extensions; reload and purge partial imports
        import importlib, typing_extensions
        importlib.reload(typing_extensions)
        for p in ("pydantic", "pydantic_core", "openai", "fabric.dataagent"):
            for m in [m for m in list(sys.modules) if m == p or m.startswith(p + ".")]:
                del sys.modules[m]
        from fabric.dataagent.evaluation import evaluate_data_agent
        return evaluate_data_agent

evaluate_data_agent = _import_sdk()
from importlib.metadata import version
SDK_VERSION = version("fabric-data-agent-sdk")
print("fabric-data-agent-sdk", SDK_VERSION)

In [ ]:
# Load registry + ground truth, apply filters ---------------------------------
agents = spark.read.table("aql_agents").toPandas()
agents = agents[agents["enabled"] == True]
if AGENT_FILTER:
    agents = agents[agents["agent_name"].str.contains(AGENT_FILTER, case=False)]
gt = spark.read.table("aql_ground_truth").toPandas()
if BANK_FILTER:
    agents = agents[agents["question_bank"] == BANK_FILTER]

if agents.empty:
    raise ValueError("No enabled agents match the filters - run 01_register_agents first.")
print(f"{len(agents)} agent(s) to evaluate, {len(gt)} ground-truth question(s) available")
display(agents[["agent_name", "workspace_name", "stage", "question_bank"]])

In [ ]:
# Evaluation loop --------------------------------------------------------------
import time, uuid
from datetime import datetime
import pandas as pd

run_rows = []
for _, a in agents.iterrows():
    bank = gt[gt["bank_name"] == a["question_bank"]]
    if SAMPLE_SIZE and SAMPLE_SIZE > 0 and len(bank) > SAMPLE_SIZE:
        bank = bank.sample(SAMPLE_SIZE, random_state=42)
    started = datetime.utcnow(); t0 = time.time()
    row = {"agent_name": str(a["agent_name"]), "workspace_name": str(a["workspace_name"]), "stage": str(a["stage"]),
           "bank_name": str(a["question_bank"]), "question_count": int(len(bank)),
           "started_at": started, "sdk_version": str(SDK_VERSION)}
    if bank.empty:
        row.update(run_id=str(uuid.uuid4()), status="skipped_no_bank", error=None,
                   finished_at=datetime.utcnow(), duration_s=0.0)
        run_rows.append(row); print(f"SKIP {a['agent_name']}: bank '{a['question_bank']}' is empty"); continue
    df_in = bank[["question", "expected_answer"]].reset_index(drop=True)
    print(f"EVAL {a['agent_name']} ({a['workspace_name']}, {a['stage']}) - {len(df_in)} question(s)...")
    try:
        eval_id = evaluate_data_agent(
            df_in,
            data_agent_name=a["agent_name"],
            workspace_name=a["workspace_name"],
            table_name=OUTPUT_TABLE,
            critic_prompt=(CRITIC_PROMPT or None),
            data_agent_stage=a["stage"],
            max_workers=int(MAX_WORKERS),
            num_query_repeats=int(NUM_QUERY_REPEATS))
        row.update(run_id=str(eval_id), status="completed", error=None)
    except Exception as ex:
        row.update(run_id=str(uuid.uuid4()), status="failed",
                   error=f"{type(ex).__name__}: {ex}"[:2000])
        print(f"  FAILED: {row['error'][:200]}")
    row.update(finished_at=datetime.utcnow(), duration_s=round(time.time() - t0, 1))
    run_rows.append(row)

# Explicit Spark schema + pure-Python records: pandas int64/float64 clash with INT/DOUBLE DDL on append
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DoubleType, TimestampType)
runs_schema = StructType([
    StructField("run_id", StringType()), StructField("agent_name", StringType()),
    StructField("workspace_name", StringType()), StructField("stage", StringType()),
    StructField("bank_name", StringType()), StructField("question_count", IntegerType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("duration_s", DoubleType()), StructField("status", StringType()),
    StructField("error", StringType()), StructField("sdk_version", StringType()),
])
records = [tuple(r.get(fld.name) for fld in runs_schema.fields) for r in run_rows]
spark.createDataFrame(records, schema=runs_schema).write.mode("append").saveAsTable("aql_eval_runs")

runs_df = pd.DataFrame(run_rows)
display(runs_df[["agent_name", "status", "question_count", "duration_s", "error"]])

In [ ]:
# Rebuild derived daily scores --------------------------------------------------
from pyspark.sql import functions as F

out = spark.read.table(OUTPUT_TABLE)
runs = spark.read.table("aql_eval_runs").select("run_id", "agent_name", "workspace_name", "bank_name", "stage")
scores = (out.join(runs, out["evaluation_id"] == runs["run_id"], "inner")
    .groupBy(F.to_date("run_timestamp").alias("eval_date"), "agent_name", "workspace_name",
             "bank_name", "stage", "evaluation_id")
    .agg(F.count("*").alias("n_questions"),
         F.sum(F.when(F.col("evaluation_judgement") == True, 1).otherwise(0)).alias("n_true"),
         F.sum(F.when(F.col("evaluation_judgement") == False, 1).otherwise(0)).alias("n_false"),
         F.sum(F.when(F.col("evaluation_judgement").isNull(), 1).otherwise(0)).alias("n_unclear"),
         F.avg("execution_time_sec").alias("avg_latency_s"),
         F.max("data_agent_version").alias("agent_version"))
    .withColumn("accuracy", F.round(F.col("n_true") / F.col("n_questions"), 4)))
scores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("aql_agent_scores")

print("aql_agent_scores rebuilt")
display(scores.orderBy(F.desc("eval_date")).limit(20).toPandas())

In [ ]:
# Detect regressions vs the previous evaluation --------------------------------
from datetime import datetime as _dt

sc = spark.read.table("aql_agent_scores").toPandas()
rn = spark.read.table("aql_eval_runs").toPandas()[["run_id", "started_at"]]
sc = sc.merge(rn, left_on="evaluation_id", right_on="run_id", how="left").sort_values("started_at")

regressions = []
for (agent, wsn), g in sc.groupby(["agent_name", "workspace_name"]):
    if len(g) < 2:
        continue
    prev, last = g.iloc[-2], g.iloc[-1]
    drop = float(prev["accuracy"]) - float(last["accuracy"])
    if drop >= float(REGRESSION_THRESHOLD):
        regressions.append({"agent_name": str(agent), "workspace_name": str(wsn),
                            "prev_accuracy": float(prev["accuracy"]), "latest_accuracy": float(last["accuracy"]),
                            "drop": round(drop, 4), "detected_at": _dt.utcnow()})

spark.sql("""CREATE TABLE IF NOT EXISTS aql_regressions (
    agent_name STRING, workspace_name STRING, prev_accuracy DOUBLE,
    latest_accuracy DOUBLE, drop DOUBLE, detected_at TIMESTAMP) USING DELTA""")
if regressions:
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
    r_schema = StructType([
        StructField("agent_name", StringType()), StructField("workspace_name", StringType()),
        StructField("prev_accuracy", DoubleType()), StructField("latest_accuracy", DoubleType()),
        StructField("drop", DoubleType()), StructField("detected_at", TimestampType())])
    recs = [tuple(r[f.name] for f in r_schema.fields) for r in regressions]
    spark.createDataFrame(recs, schema=r_schema).write.mode("append").saveAsTable("aql_regressions")
    for r in regressions:
        print(f"REGRESSION: {r['agent_name']} accuracy {r['prev_accuracy']:.0%} -> {r['latest_accuracy']:.0%} (drop {r['drop']:.0%})")
else:
    print("No regressions detected.")

# Pointer recommendations so the report's "What to fix" page is always actionable
pointer_recs = []
_now = _dt.utcnow()
for rr in run_rows:
    if rr["status"] == "failed":
        pointer_recs.append((rr["run_id"], rr["agent_name"], "agent_error", int(rr["question_count"]),
                             ("The evaluation call failed: " + str(rr["error"])[:280] +
                              " Verify the agent's workspace is on an ACTIVE paid F SKU capacity and you have access to the agent."), _now))
for rg in regressions:
    _rid = next((rr["run_id"] for rr in run_rows
                 if rr["agent_name"] == rg["agent_name"] and rr["status"] == "completed"), "")
    pointer_recs.append((_rid, rg["agent_name"], "regression_detected", 0,
                         (f"Accuracy dropped {rg['prev_accuracy']:.0%} -> {rg['latest_accuracy']:.0%}. "
                          f"Run 03_analyze_failures with RUN_ID='{_rid}' for per-question diagnosis, "
                          "generated-query inspection and the config-drift check."), _now))
if pointer_recs:
    spark.sql("""CREATE TABLE IF NOT EXISTS aql_recommendations (
        run_id STRING, agent_name STRING, diagnosis STRING, n_questions INT,
        recommendation STRING, created_at TIMESTAMP) USING DELTA""")
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
    _prs = StructType([StructField("run_id", StringType()), StructField("agent_name", StringType()),
                       StructField("diagnosis", StringType()), StructField("n_questions", IntegerType()),
                       StructField("recommendation", StringType()), StructField("created_at", TimestampType())])
    spark.createDataFrame(pointer_recs, schema=_prs).write.mode("append").saveAsTable("aql_recommendations")
    print(f"{len(pointer_recs)} recommendation pointer(s) written for the What-to-fix page.")

In [ ]:
# Refresh the quality-lab semantic model so reports reflect this run -----------
try:
    import sempy.fabric as sf
    sf.refresh_dataset("agent_quality_lab_model", refresh_type="full")
    print("Semantic model refresh triggered.")
except Exception as ex:
    print(f"Model refresh skipped (model missing or refresh failed): {ex}")

In [ ]:
# Summary + pipeline exit value -------------------------------------------------
import json
summary = {
    "agents_evaluated": int((runs_df["status"] == "completed").sum()),
    "agents_failed": int((runs_df["status"] == "failed").sum()),
    "agents_skipped": int((runs_df["status"] == "skipped_no_bank").sum()),
    "run_ids": runs_df.loc[runs_df["status"] == "completed", "run_id"].tolist(),
    "regressions": regressions,
}
print(json.dumps(summary, indent=2))
try:
    import notebookutils
    notebookutils.notebook.exit(json.dumps(summary))
except Exception:
    pass  # not running as a notebook job
if FAIL_ON_REGRESSION and regressions:
    raise RuntimeError(f"{len(regressions)} agent(s) regressed beyond threshold {REGRESSION_THRESHOLD}")